# Exploración: páginas de Facebook de empresas

1. **RUC → nombres** con los extractores actuales (apis.net.pe y SUNAT).
2. **Lectura** de páginas conocidas, sin login.
3. **Búsqueda** de la página en Google, DuckDuckGo y Bing con varias consultas.

Navegador normal: sin login, sin disfrazarlo y sin resolver captchas. Ante un bloqueo, esa fuente **se detiene**.
Limpia las salidas antes de hacer commit.

In [ ]:
import os, sys, re, json, random, asyncio, unicodedata
from urllib.parse import urlsplit, parse_qs, quote_plus, unquote

if not os.path.isdir("app"):
    os.chdir(os.path.abspath(".."))  # raíz del repo
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv()
from playwright.async_api import async_playwright
from app.infrastructure.outbound.extractors.shared.browser import run_isolated  # solo el bucle asyncio
from app.infrastructure.outbound.extractors.profile.ruc_adapter import ApisNetPeRucAdapter
from app.infrastructure.outbound.extractors.sunat.ruc_adapter import SunatRucAdapter

# RUC -> página esperada
CASOS = {"20601590175": "internetapurimacamericaservices", "20603687478": "codideep"}
PAGINAS = ["InternetApurimacAmericaServices", "codideep", "estapaginanoexiste98431pe"]

In [ ]:
# 1) RUC -> nombres y dirección
async def datos_ruc(ruc):
    perfil, sunat = await asyncio.gather(ApisNetPeRucAdapter().extract(ruc), SunatRucAdapter().extract(ruc))
    p, s = perfil["data"], sunat["data"]
    return {"ruc": ruc, "estado": [perfil["audit"].status.value, sunat["audit"].status.value],
            "razon_social": p.legal_name if p else None,
            "nombre_comercial": s.commercial_name if s else None,
            "direccion": (s.fiscal_address if s else None) or (p.address if p else None),
            "departamento": p.department if p else None, "provincia": p.province if p else None,
            "actividad": list(s.economic_activities) if s else []}

empresas = {}
for ruc in CASOS:
    empresas[ruc] = await datos_ruc(ruc)
    print(json.dumps(empresas[ruc], ensure_ascii=False, indent=1))

In [ ]:
JS_FB = """() => {
  const metas = {};
  for (const e of document.querySelectorAll('meta[property], meta[name]'))
    metas[e.getAttribute('property') || e.getAttribute('name')] = e.content;
  return {url: location.href, title: document.title, metas, body: document.body ? document.body.innerText : '',
    password: !!document.querySelector('input[type=password]'),
    enlaces: [...document.querySelectorAll('a[href]')].map(a => ({href: a.href, texto: a.innerText.slice(0, 80)})),
    json: [...document.querySelectorAll('script[type="application/json"]')].map(s => s.textContent)};
}"""

async def con_navegador(tarea, headless=True):
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(channel="chrome", headless=headless)
        try:
            return await tarea(browser)
        finally:
            await browser.close()

async def leer_fb(browser, usuario):
    context = await browser.new_context(locale="es-PE", viewport={"width": 1366, "height": 768})
    try:
        page = await context.new_page()
        r = await page.goto(f"https://www.facebook.com/{usuario}/", wait_until="domcontentloaded", timeout=45000)
        await page.wait_for_timeout(6000)
        p = await page.evaluate(JS_FB)
        p["status"] = r.status if r else None
        return p
    finally:
        await context.close()

async def leer_todas(browser):
    res = {}
    for i, u in enumerate(PAGINAS):
        if i:
            await asyncio.sleep(random.uniform(10, 15))
        res[u] = await leer_fb(browser, u)
        p = res[u]
        print("=" * 15, u, "| HTTP", p["status"], "|", p["url"], "|", p["title"])
        for k in ("og:title", "og:description", "og:url", "og:image", "al:android:url"):
            print("  ", k, "=", (p["metas"].get(k) or "")[:260])
        print(p["body"][:1300])
        if "/login" in urlsplit(p["url"]).path or "checkpoint" in p["url"]:
            print("Bloqueo: se detiene."); break
    return res

# 2) Lectura de páginas conocidas
paginas = await run_isolated(lambda: con_navegador(leer_todas))

In [ ]:
NO_PAGINA = {"login", "sharer", "share", "dialog", "watch", "groups", "events", "marketplace", "help", "policies",
             "privacy", "hashtag", "photo", "photo.php", "story.php", "permalink.php", "reel", "reels", "videos",
             "public", "pages", "l.php", "plugins", "tr", "business", "legal", "directory", "media", "posts", "people"}

def pagina_fb(url):
    """Identificador de página en una URL de Facebook; None si no es una página."""
    try:
        u = urlsplit(url)
    except ValueError:
        return None
    if not re.fullmatch(r"(?:[\w-]+\.)?facebook\.com", u.hostname or ""):
        return None
    partes = [x for x in u.path.split("/") if x]
    if not partes:
        return None
    if partes[0] == "profile.php":
        return "id:" + (parse_qs(u.query).get("id") or [""])[0]
    if partes[0] in ("p", "people") and len(partes) > 1:
        return "/".join(partes[:3]).lower()
    return None if partes[0].lower() in NO_PAGINA else partes[0].lower()

def destino(url):
    """Deshace las redirecciones de los buscadores."""
    u = urlsplit(url)
    q = parse_qs(u.query)
    if "duckduckgo.com" in (u.hostname or "") and "uddg" in q:
        return q["uddg"][0]
    if "google." in (u.hostname or "") and u.path == "/url" and ("q" in q or "url" in q):
        return (q.get("q") or q.get("url"))[0]
    if "bing.com" in (u.hostname or "") and u.path.startswith("/ck/") and "u" in q:
        import base64
        c = q["u"][0][2:]
        try:
            return base64.urlsafe_b64decode(c + "=" * (-len(c) % 4)).decode()
        except Exception:
            return url
    return url

BUSCADORES = {
    "google": ("https://www.google.com/search?hl=es&gl=pe&q=", "#search a[href], #rso a[href]"),
    "duckduckgo": ("https://html.duckduckgo.com/html/?kl=pe-es&q=", "a.result__a"),
    "bing": ("https://www.bing.com/search?setlang=es&cc=PE&q=", "#b_results h2 a"),
}
BLOQUEO = re.compile(r"unusual traffic|tráfico inusual|no eres un robot|not a robot|captcha|anomaly|select all squares|"
                     r"bots use duckduckgo|one last step|un último paso|verify you are human", re.I)

def forma_societaria_fuera(nombre):
    n = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode().upper()
    n = re.sub(r"\b(EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA|SOCIEDAD ANONIMA CERRADA|SOCIEDAD ANONIMA ABIERTA|"
               r"SOCIEDAD COMERCIAL DE RESPONSABILIDAD LIMITADA|SOCIEDAD ANONIMA)\b.*$", "", n)
    n = re.sub(r"[\s.-]*\b(S\.?\s?A\.?\s?C|E\.?\s?I\.?\s?R\.?\s?L|S\.?\s?R\.?\s?L|S\.?\s?A\.?\s?A|S\.?\s?A)\.?\s*$", "", n)
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", n).split())

def consultas(e):
    nombre = forma_societaria_fuera(e["razon_social"] or "")
    c = {"ruc": f'"{e["ruc"]}"', "ruc+fb": f'{e["ruc"]} facebook', "nombre+fb": f"{nombre} facebook",
         "site": f"{nombre} site:facebook.com"}
    com = e["nombre_comercial"]
    if com and forma_societaria_fuera(com) not in ("", nombre):
        c["comercial+fb"] = f"{forma_societaria_fuera(com)} facebook"
    return c

async def buscar(browser, motor):
    base, sel = BUSCADORES[motor]
    out = []
    context = await browser.new_context(locale="es-PE", viewport={"width": 1366, "height": 768})
    try:
        page = await context.new_page()
        for ruc, e in empresas.items():
            for tipo, q in consultas(e).items():
                if out:
                    await asyncio.sleep(random.uniform(8, 14))
                r = await page.goto(base + quote_plus(q), wait_until="domcontentloaded", timeout=45000)
                await page.wait_for_timeout(2500)
                texto = await page.evaluate("document.body.innerText")
                urls = await page.evaluate("s => [...document.querySelectorAll(s)].map(a => a.href)", sel)
                urls = list(dict.fromkeys(destino(u) for u in urls))
                urls = [u for u in urls if u.startswith("http") and not re.search(r"//([\w-]+\.)*google\.", u)]
                fb = list(dict.fromkeys(p for p in map(pagina_fb, urls) if p))
                bloqueado = (r and r.status in (202, 403, 429)) or "/sorry/" in page.url or (not urls and bool(BLOQUEO.search(texto)))
                pos = fb.index(CASOS[ruc]) + 1 if CASOS[ruc] in fb else None
                out.append({"motor": motor, "ruc": ruc, "tipo": tipo, "q": q, "http": r.status if r else None,
                            "bloqueado": bool(bloqueado), "urls": urls[:12], "fb": fb, "pos": pos})
                print(f"{motor:11} {ruc} {tipo:13} HTTP {out[-1]['http']} res={len(urls):2} fb={fb[:5]} esperada en pos={pos}"
                      + ("  BLOQUEADO" if bloqueado else ""))
                if bloqueado:
                    print("  ", " ".join(texto.split())[:200]); return out
    except Exception as ex:
        print(motor, "ERROR", type(ex).__name__, str(ex)[:150])
    finally:
        await context.close()
    return out

In [ ]:
# 3) Búsqueda (ventana visible)
async def buscar_todos(browser):
    return sum([await buscar(browser, m) for m in BUSCADORES], [])

busquedas = await run_isolated(lambda: con_navegador(buscar_todos, headless=False))

In [ ]:
# 4) Bing: consultas con ciudad, rubro y direccion (ronda corta)
EXTRA = {
    "20601590175": ["America Services and Solutions Abancay facebook", "America Services Abancay internet facebook",
                    "internet fibra optica Abancay facebook", "Jr. Santa Rosa 116 Abancay internet"],
    "20603687478": ["Codideep Abancay software facebook"],
}
async def ronda2(browser):
    out = []
    context = await browser.new_context(locale="es-PE", viewport={"width": 1366, "height": 768})
    page = await context.new_page()
    base, sel = BUSCADORES["bing"]
    for ruc, qs in EXTRA.items():
        for q in qs:
            await asyncio.sleep(random.uniform(8, 14))
            r = await page.goto(base + quote_plus(q), wait_until="domcontentloaded", timeout=45000)
            await page.wait_for_timeout(2500)
            urls = list(dict.fromkeys(destino(u) for u in await page.evaluate("s => [...document.querySelectorAll(s)].map(a => a.href)", sel)))
            fb = [p for p in map(pagina_fb, urls) if p]
            out.append({"ruc": ruc, "q": q, "urls": urls, "fb": fb})
            print(q, "| fb:", fb, "| esperada:", CASOS[ruc] in fb)
            for u in urls[:8]: print("     ", u[:110])
            if not urls and BLOQUEO.search(await page.evaluate("document.body.innerText")):
                print("BLOQUEADO"); return out
    return out

ronda = await run_isolated(lambda: con_navegador(ronda2, headless=False))

In [ ]:
# 5) Web oficial -> Facebook y RUC
import httpx
async def revisar_web(url, ruc):
    try:
        async with httpx.AsyncClient(follow_redirects=True, timeout=20) as c:
            r = await c.get(url)
        patron = r'https?://[\w.-]*facebook\.com/[\w./?=&%-]+'
        fb = sorted({p for p in map(pagina_fb, re.findall(patron, r.text)) if p})
        print(url, r.status_code, "| enlaces a Facebook:", fb, "| contiene RUC:", ruc in r.text)
    except Exception as ex:
        print(url, "ERROR", type(ex).__name__)
for u in ("https://codideep.com/", "https://eirl.codideep.com/"):
    await revisar_web(u, "20603687478")

In [ ]:
# 6) Usernames probables derivados del nombre (pocos)
async def probar(browser):
    res = {}
    for u in ("americaservices", "americaservicesandsolutions", "americaservicesabancay"):
        await asyncio.sleep(random.uniform(10, 15))
        p = res[u] = await leer_fb(browser, u)
        print(u, "|", p["url"], "|", p["title"], "|", (p["metas"].get("og:description") or "")[:120])
    return res
probables = await run_isolated(lambda: con_navegador(probar))

In [ ]:
# 7) Extractor definitivo (página + información + transparencia), sin base de datos
from app.infrastructure.outbound.extractors.facebook.page_adapter import FacebookPageAdapter

r = await FacebookPageAdapter().extract("https://www.facebook.com/codideep")
print(r["audit"].status.value, r["audit"].details)
r["data"]